# Reactor criticality

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/reactor_criticality)

Is a slab of fissile material critical? One-group neutron diffusion turns the question into a generalised eigenvalue problem $M\phi = \tfrac1k F\phi$ on a grid. Power iteration — repeatedly solving for the next neutron generation — converges to the fundamental flux mode and the multiplication factor $k_{eff}$, and a batched eigen solve over many slab widths finds the critical size.

**Vector form**

$$-D\,\nabla^2\phi + \Sigma_a\,\phi = \frac{1}{k}\,\nu\Sigma_f\,\phi,\qquad k = \frac{\nu\Sigma_f}{\Sigma_a + D B^2},\quad B = \frac{\pi}{L}$$

**Index (tensor) form**

$$M_{ij}\,\phi_j = \frac1k\,F_{ij}\,\phi_j,\quad M_{ij} = -D\,\Delta_{ij} + \Sigma_a\delta_{ij},\quad F_{ij} = \nu\Sigma_f\,\delta_{ij},\qquad M_{ij}\,\phi^{(n+1)}_j = \frac{F_{ij}\,\phi^{(n)}_j}{k^{(n)}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_i$ | `grid(width, N)` | (N,) | interior points; $\phi = 0$ at the (extrapolated) slab faces |
| $\Delta_{ij}$ | `laplacian(N, h)` | (N, N) | second-difference matrix $(1, -2, 1)/h^2$ |
| $M_{ij}$ | `loss_operator(lap, D, sigma_a)` | (N, N) | neutrons lost by leakage and absorption |
| $F_{ij}$ | `production_operator(N, nu_sigma_f)` | (N, N) | neutrons born in fission |
| $F_{ij}\phi_j$ | `einsum('ij,j->i', f, phi)` | (N,) | repeated $j$ ⇒ summed: the fission source of the next generation |
| $\phi^{(n+1)}_j$ | `torch.linalg.solve(m, source / k)` | (N,) | one power-iteration step |
| $k_{eff}$ | `eigvals(solve(m, f)).real.max()` | (S,) | direct solve, batched over $S$ slab widths |

$k$ is the ratio of neutrons in one generation to the previous one. Each power-iteration step is one generation: the higher spatial modes shrink by the factor $k_2/k_1 = (\Sigma_a + DB^2)/(\Sigma_a + 4DB^2)$ every step, so the lopsided start relaxes to the symmetric $\sin(\pi x/L)$ shape. A thin slab leaks too many neutrons ($k < 1$); above the critical width $L_c = \pi\sqrt{D/(\nu\Sigma_f - \Sigma_a)}$ the chain reaction grows. The default one-group constants are illustrative of a thermal reactor ($k_\infty \approx 1.06$). Simplifications: one energy group, a homogeneous bare slab with no reflector, $L$ measured between extrapolated faces, and no delayed neutrons or feedback — this is a static eigenvalue calculation, not a time-dependent one.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/reactor_criticality/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import analytic_k, critical_width, loss_operator, production_operator


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    width, n = params["width"], params["n_grid"]
    d, sa, nsf = params["diffusion"], params["sigma_a"], params["nu_sigma_f"]
    x, h = tensors.grid(width, n, dev)  # (N,) cm
    m = loss_operator(tensors.laplacian(n, h, dev), d, sa)  # (N, N)
    f = production_operator(n, nsf, dev)  # (N, N)

    ks, fluxes = solve.power_iteration(m, f, tensors.initial_guess(x, width), params["n_iter"])
    k_direct = solve.k_eigen(m, f).item()
    k_exact = analytic_k(width, d, sa, nsf)

    l_crit = critical_width(d, sa, nsf)
    widths = torch.linspace(0.4 * width, 2.0 * width, 40, dtype=torch.float64, device=dev)  # (S,)
    k_widths = solve.k_versus_width(widths, 100, d, sa, nsf)  # (S,)

    print(f"k∞ = νΣf/Σa = {nsf / sa:.4f}   critical width L_c = {l_crit:.2f} cm")
    print(f"slab L = {width:.1f} cm:  power iteration k = {ks[-1].item():.6f}  ({params['n_iter']} iterations)")
    print(f"                eigen solve      k = {k_direct:.6f}")
    print(f"                analytic νΣf/(Σa + D(π/L)²) = {k_exact:.6f}")
    state = "supercritical" if k_exact > 1 else "subcritical" if k_exact < 1 else "critical"
    print(f"→ the slab is {state}")
    return plot.animate(x, width, ks, fluxes, k_exact, widths, k_widths, l_crit, out_dir / "reactor_criticality.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/reactor_criticality/equation.py
r"""Physics → tensor translation.

One-group neutron diffusion in a bare slab of width L: neutrons diffuse (D), are absorbed (Σ_a) and fission
produces ν new neutrons per fission (νΣ_f). A steady state exists only if the production is scaled by 1/k:

    -D \nabla^2 φ + Σ_a φ = \frac{1}{k} νΣ_f φ,        φ = 0 at x = 0 and x = L (extrapolated edges).

k_eff > 1: supercritical, = 1: critical, < 1: subcritical. On a grid x_i with the second-difference matrix
Δ_{ij} this becomes a generalised eigenproblem with a loss matrix M and a production matrix F:

    M_{ij} φ_j = \frac{1}{k} F_{ij} φ_j,        M_{ij} = -D Δ_{ij} + Σ_a δ_{ij},    F_{ij} = νΣ_f δ_{ij}.

Power iteration (what reactor codes do): solve M φ^{(n+1)} = F φ^{(n)} / k^{(n)}, then
k^{(n+1)} = k^{(n)} Σ_i (Fφ^{(n+1)})_i / Σ_i (Fφ^{(n)})_i.  The analytic answer for the bare slab is

    k = \frac{νΣ_f}{Σ_a + D B^2},    B = π / L,    φ(x) ∝ \sin(π x / L).
"""

import math

import torch


def loss_operator(lap: torch.Tensor, d: float, sigma_a: float) -> torch.Tensor:
    r"""M_{ij} = −D Δ_{ij} + Σ_a δ_{ij}  (leakage + absorption).   lap: (..., N, N) → (..., N, N)"""
    eye = torch.eye(lap.shape[-1], dtype=lap.dtype, device=lap.device)
    return -d * lap + sigma_a * eye


def production_operator(n: int, nu_sigma_f: float, device: torch.device) -> torch.Tensor:
    r"""F_{ij} = νΣ_f δ_{ij}  (fission source).   → (N, N)"""
    return nu_sigma_f * torch.eye(n, dtype=torch.float64, device=device)


def power_step(m: torch.Tensor, f: torch.Tensor, phi: torch.Tensor, k: float) -> tuple[torch.Tensor, float]:
    r"""φ' = M^{-1} F φ / k,   k' = k · Σ_i (Fφ')_i / Σ_i (Fφ)_i.   phi: (N,) → ((N,), float)"""
    source = torch.einsum("ij,j->i", f, phi)  # F_{ij} φ_j
    phi_new = torch.linalg.solve(m, source / k)  # M_{ij} φ'_j = F_{ij} φ_j / k
    k_new = k * torch.einsum("ij,j->", f, phi_new).item() / source.sum().item()
    return phi_new, k_new


def analytic_k(width: torch.Tensor | float, d: float, sigma_a: float, nu_sigma_f: float):
    r"""k = νΣ_f / (Σ_a + D B²) with buckling B = π/L."""
    return nu_sigma_f / (sigma_a + d * (math.pi / width) ** 2)


def critical_width(d: float, sigma_a: float, nu_sigma_f: float) -> float:
    r"""k = 1  ⇔  B² = (νΣ_f − Σ_a)/D  ⇔  L_c = π \sqrt{D / (νΣ_f − Σ_a)}  (∞ if νΣ_f ≤ Σ_a)."""
    excess = nu_sigma_f - sigma_a
    return math.pi * math.sqrt(d / excess) if excess > 0 else math.inf


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/reactor_criticality/tensors.py
"""Build the concrete tensors: the slab grid, the discrete Laplacian and the starting flux guess."""

import torch


def grid(width: float, n: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """Interior points x_i = h … L − h (the flux is 0 at both edges): (N,), spacing h."""
    h = width / (n + 1)
    return torch.arange(1, n + 1, dtype=torch.float64, device=device) * h, h


def laplacian(n: int, h: torch.Tensor | float, device: torch.device) -> torch.Tensor:
    """Δ_ij φ_j ≈ φ''(x_i): tridiagonal (1, −2, 1)/h². With a tensor of S spacings h → (S, N, N), else (N, N)."""
    main = torch.full((n,), -2.0, dtype=torch.float64, device=device)
    off = torch.ones(n - 1, dtype=torch.float64, device=device)
    stencil = torch.diag(main) + torch.diag(off, 1) + torch.diag(off, -1)  # (N, N)
    if isinstance(h, torch.Tensor):
        return stencil / h[:, None, None] ** 2
    return stencil / h**2


def initial_guess(x: torch.Tensor, width: float) -> torch.Tensor:
    """A deliberately lopsided start, φ⁽⁰⁾ = 0.2 + x/L, so we can watch it relax to the fundamental mode: (N,)."""
    return 0.2 + x / width


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/reactor_criticality/solve.py
"""k_eff two ways: power iteration (the reactor-physics workhorse) and a direct eigenvalue solve."""

import torch

from problems._common.progress import steps

from .equation import loss_operator, power_step, production_operator
from .tensors import laplacian


def power_iteration(
    m: torch.Tensor, f: torch.Tensor, phi0: torch.Tensor, n_iter: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (k history (I+1,), flux history (I+1, N)); each flux is normalised to peak 1 for plotting."""
    phi, k = phi0, 1.0
    ks, fluxes = [k], [phi / phi.max()]
    for _ in steps(n_iter, "power iterations"):
        phi, k = power_step(m, f, phi, k)
        ks.append(k)
        fluxes.append(phi / phi.max())
    return torch.tensor(ks, dtype=torch.float64), torch.stack(fluxes)


def k_eigen(m: torch.Tensor, f: torch.Tensor) -> torch.Tensor:
    """Largest eigenvalue of M⁻¹F (the fundamental mode's k). Works batched: (..., N, N) → (...)."""
    return torch.linalg.eigvals(torch.linalg.solve(m, f)).real.max(dim=-1).values


def k_versus_width(widths: torch.Tensor, n: int, d: float, sigma_a: float, nu_sigma_f: float) -> torch.Tensor:
    """k_eff for a whole batch of slab widths at once: one (S, N, N) stack of loss matrices → (S,)."""
    h = widths / (n + 1)  # (S,) grid spacing of each slab
    m = loss_operator(laplacian(n, h, widths.device), d, sigma_a)  # (S, N, N)
    f = production_operator(n, nu_sigma_f, widths.device).expand_as(m)  # (S, N, N)
    return k_eigen(m, f)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/reactor_criticality/plot.py
"""Watch power iteration converge: flux shape, k per iteration, and k_eff against slab width."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    x: torch.Tensor,
    width: float,
    ks: torch.Tensor,
    fluxes: torch.Tensor,
    k_exact: float,
    widths: torch.Tensor,
    k_widths: torch.Tensor,
    l_crit: float,
    path: Path,
) -> Path:
    x_np, phi = x.cpu().numpy(), fluxes.cpu().numpy()  # (N,), (I+1, N)
    k_np = ks.cpu().numpy()
    it = np.arange(len(k_np))

    fig, (ax_f, ax_k, ax_w) = plt.subplots(1, 3, figsize=(10.5, 3.6))
    ax_f.plot(x_np, np.sin(np.pi * x_np / width), color="#9ca3af", lw=2, ls="--", label=r"$\sin(\pi x/L)$")
    (start,) = ax_f.plot(x_np, phi[0], color="#93c5fd", lw=1, label="start guess")
    (line,) = ax_f.plot([], [], color="#2563eb", lw=2, label="iterate")
    ax_f.set_xlim(0, width)
    ax_f.set_ylim(0, 1.15)
    ax_f.set_xlabel("x (cm)")
    ax_f.set_ylabel("flux φ (peak = 1)")
    ax_f.legend(loc="lower center", fontsize=7)
    title = ax_f.set_title("")

    ax_k.axhline(k_exact, color="#9ca3af", lw=2, ls="--", label="analytic k")
    (k_line,) = ax_k.plot([], [], "o-", color="#dc2626", ms=3, lw=1, label="power iteration")
    ax_k.set_xlim(0, it[-1])
    lo, hi = min(k_np.min(), k_exact), max(k_np.max(), k_exact)
    ax_k.set_ylim(lo - 0.1 * (hi - lo) - 1e-3, hi + 0.1 * (hi - lo) + 1e-3)
    ax_k.set_xlabel("iteration n")
    ax_k.set_ylabel("$k^{(n)}$")
    ax_k.legend(loc="upper right", fontsize=7)

    w_np, kw_np = widths.cpu().numpy(), k_widths.cpu().numpy()
    ax_w.plot(w_np, kw_np, color="#2563eb", lw=2)
    ax_w.axhline(1.0, color="#9ca3af", lw=1)
    if np.isfinite(l_crit) and w_np[0] <= l_crit <= w_np[-1]:
        ax_w.axvline(l_crit, color="#f59e0b", lw=1, ls="--")
        ax_w.text(l_crit, kw_np.min(), f" critical\n L = {l_crit:.1f} cm", color="#b45309", fontsize=8, va="bottom")
    ax_w.plot([width], [k_exact], "o", color="#dc2626")
    ax_w.set_xlabel("slab width L (cm)")
    ax_w.set_ylabel("$k_{eff}$")
    ax_w.set_title("eigen solve, batched over L", fontsize=9)

    def update(step: int):
        line.set_data(x_np, phi[step])
        start.set_alpha(1.0 if step < 3 else 0.4)
        k_line.set_data(it[: step + 1], k_np[: step + 1])
        title.set_text(f"iteration {step}:  k = {k_np[step]:.5f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(k_np), 61), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'width': 60.0,
    'diffusion': 0.9,
    'sigma_a': 0.066,
    'nu_sigma_f': 0.07,
    'n_iter': 60,
    'n_grid': 200
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.reactor_criticality")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.reactor_criticality.main")
problem_dir = Path("problems/nuclear/reactor_criticality")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
